# Geração de Tabela LaTeX a partir de arquivos .txt

Este notebook lê todos os arquivos `.txt` da pasta `input`, processa os dados e gera uma tabela em formato LaTeX para ser utilizada no relatório.

## 1. Importar bibliotecas necessárias
Vamos importar as bibliotecas pandas e os para manipulação de arquivos e dados.

In [92]:
import pandas as pd
import os
from pathlib import Path

## 2. Listar arquivos .txt na pasta input
Vamos listar todos os arquivos `.csv` presentes na pasta `input`.

In [93]:
from pathlib import Path
import os

# Caminho correto para a pasta de dados (relativo ao notebook)
input_dir = Path(os.getcwd()) / 'dados'
if not input_dir.exists():
    input_dir = Path('./dados').resolve()
print('Diretório de trabalho atual:', os.getcwd())
print('Caminho usado para a pasta de dados:', input_dir)

# Listar arquivos .csv (minúsculo e maiúsculo)
arquivos_csv = list(input_dir.glob('*.csv')) + list(input_dir.glob('*.CSV'))
print('Arquivos .csv encontrados:', arquivos_csv)

# Listar todo o conteúdo da pasta para depuração
print('Conteúdo da pasta dados:')
for f in input_dir.iterdir():
    print(f, '->', 'arquivo' if f.is_file() else 'pasta')


Diretório de trabalho atual: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\src
Caminho usado para a pasta de dados: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\src\dados
Arquivos .csv encontrados: [WindowsPath('d:/GitHub/DoutoradoCefet/PrincipioModelagemMatematica/pratica01/src/dados/acuracia_com_resistencia_c_0_01.csv'), WindowsPath('d:/GitHub/DoutoradoCefet/PrincipioModelagemMatematica/pratica01/src/dados/acuracia_com_resistencia_c_0_1.csv'), WindowsPath('d:/GitHub/DoutoradoCefet/PrincipioModelagemMatematica/pratica01/src/dados/acuracia_com_resistencia_c_10.csv'), WindowsPath('d:/GitHub/DoutoradoCefet/PrincipioModelagemMatematica/pratica01/src/dados/acuracia_com_resistencia_c_2.csv'), WindowsPath('d:/GitHub/DoutoradoCefet/PrincipioModelagemMatematica/pratica01/src/dados/acuracia_sem_resistencia.csv'), WindowsPath('d:/GitHub/DoutoradoCefet/PrincipioModelagemMatematica/pratica01/src/dados/saida_queda_livre_dt_0_001.csv'), WindowsPath('d:/GitHub/Dou

## 3. Ler e processar arquivos .csv
Vamos ler o conteúdo de cada arquivo `.csv` e armazenar os dados em uma lista.

In [94]:
dados = []
nomes_arquivos = []
cabecalho = None

for arquivo in arquivos_csv:
    print(f"Lendo arquivo: {arquivo.name}")
    try:
        import pandas as pd
        df = pd.read_csv(arquivo)
        print("Primeiras linhas:")
        print(df.head())
        if cabecalho is None:
            cabecalho = list(df.columns)
        for _, row in df.iterrows():
            dados.append(row.tolist())
            nomes_arquivos.append(arquivo.name)
    except Exception as e:
        print(f'Erro ao ler {arquivo.name}: {e}')

print(f'Total de linhas lidas: {len(dados)}')
if cabecalho:
    print('Cabeçalho detectado:', cabecalho)
else:
    print('Nenhum cabeçalho detectado!')

Lendo arquivo: acuracia_com_resistencia_c_0_01.csv
Primeiras linhas:
      dt  tempo_final  v_final_num  v_final_analitico  erro_relativo
0  0.001        4.483   -44.978669          44.978899       1.999995
1  0.011        4.488   -45.027667          45.030200       1.999944
2  0.021        4.494   -45.086922          45.091763       1.999893
3  0.031        4.526   -45.412966          45.420165       1.999841
4  0.041        4.510   -45.246465          45.255951       1.999790
Lendo arquivo: acuracia_com_resistencia_c_0_1.csv
Primeiras linhas:
      dt  tempo_final  v_final_num  v_final_analitico  erro_relativo
0  0.001        4.201   -51.216224          51.219360       1.999939
1  0.011        4.213   -51.364039          51.398651       1.999327
2  0.021        4.221   -51.452093          51.518297       1.998715
3  0.031        4.247   -51.809299          51.907811       1.998102
4  0.041        4.264   -52.032108          52.163041       1.997490
Lendo arquivo: acuracia_com_resiste

## 4. Criar DataFrame com os dados
Agora vamos montar um DataFrame com os dados extraídos dos arquivos.

In [95]:
# Consolida todos os dados dos arquivos CSV em um único DataFrame e adiciona o nome do arquivo de origem
# Agora com relatório detalhado de sucesso/falha para cada arquivo
dados = []
nomes_arquivos = []
cabecalho = None
arquivos_lidos = 0
arquivos_ignorados = 0
arquivos_com_erro = []
arquivos_colunas_incompativeis = []

print(f"Total de arquivos CSV encontrados: {len(arquivos_csv)}")
for arquivo in arquivos_csv:
    print(f"Lendo arquivo: {arquivo.name}")
    try:
        df_temp = pd.read_csv(arquivo)
        print("Primeiras linhas:")
        print(df_temp.head())
        if cabecalho is None:
            cabecalho = list(df_temp.columns)
        # Verifica se o número de colunas bate com o cabeçalho
        linhas_validas = [row.tolist() for _, row in df_temp.iterrows() if len(row) == len(cabecalho)]
        if len(linhas_validas) < len(df_temp):
            arquivos_colunas_incompativeis.append(arquivo.name)
            print(f"AVISO: {arquivo.name} possui linhas com número de colunas diferente do cabeçalho e foram ignoradas.")
        if linhas_validas:
            dados.extend(linhas_validas)
            nomes_arquivos.extend([arquivo.name]*len(linhas_validas))
            arquivos_lidos += 1
        else:
            arquivos_ignorados += 1
            print(f"AVISO: {arquivo.name} não teve nenhuma linha válida e foi ignorado.")
    except Exception as e:
        arquivos_com_erro.append((arquivo.name, str(e)))
        print(f'ERRO ao ler {arquivo.name}: {e}')

print(f'\nResumo da leitura:')
print(f'- Arquivos lidos com sucesso: {arquivos_lidos}')
print(f'- Arquivos ignorados (sem linhas válidas): {arquivos_ignorados}')
print(f'- Arquivos com erro de leitura: {len(arquivos_com_erro)}')
if arquivos_com_erro:
    for nome, erro in arquivos_com_erro:
        print(f'  {nome}: {erro}')
if arquivos_colunas_incompativeis:
    print(f'- Arquivos com linhas ignoradas por colunas incompatíveis: {arquivos_colunas_incompativeis}')

if cabecalho and dados:
    df = pd.DataFrame(dados, columns=cabecalho)
    df['arquivo_origem'] = nomes_arquivos
    display(df)
else:
    print('Nenhum dado consolidado. Verifique os avisos acima para detalhes.')

Total de arquivos CSV encontrados: 70
Lendo arquivo: acuracia_com_resistencia_c_0_01.csv
Primeiras linhas:
      dt  tempo_final  v_final_num  v_final_analitico  erro_relativo
0  0.001        4.483   -44.978669          44.978899       1.999995
1  0.011        4.488   -45.027667          45.030200       1.999944
2  0.021        4.494   -45.086922          45.091763       1.999893
3  0.031        4.526   -45.412966          45.420165       1.999841
4  0.041        4.510   -45.246465          45.255951       1.999790
Lendo arquivo: acuracia_com_resistencia_c_0_1.csv
Primeiras linhas:
      dt  tempo_final  v_final_num  v_final_analitico  erro_relativo
0  0.001        4.201   -51.216224          51.219360       1.999939
1  0.011        4.213   -51.364039          51.398651       1.999327
2  0.021        4.221   -51.452093          51.518297       1.998715
3  0.031        4.247   -51.809299          51.907811       1.998102
4  0.041        4.264   -52.032108          52.163041       1.9974

,dt,tempo_final,v_final_num,v_final_analitico,erro_relativo,arquivo_origem
0,0.001,4.483,-44.978669,44.978899,1.999995,acuracia_com_resistencia_c_0_01.csv
1,0.011,4.488,-45.027667,45.030200,1.999944,acuracia_com_resistencia_c_0_01.csv
2,0.021,4.494,-45.086922,45.091763,1.999893,acuracia_com_resistencia_c_0_01.csv
3,0.031,4.526,-45.412966,45.420165,1.999841,acuracia_com_resistencia_c_0_01.csv
4,0.041,4.510,-45.246465,45.255951,1.999790,acuracia_com_resistencia_c_0_01.csv
...,...,...,...,...,...,...
995,0.951,5.706,-55.975860,-55.975860,0.000000,acuracia_sem_resistencia.csv
996,0.961,5.766,-56.564460,-56.564460,0.000000,acuracia_sem_resistencia.csv
997,0.971,5.826,-57.153060,-57.153060,0.000000,acuracia_sem_resistencia.csv
998,0.981,5.886,-57.741660,-57.741660,0.000000,acuracia_sem_resistencia.csv


In [96]:
# Diagnóstico: mostrar conteúdo dos arquivos problemáticos
from IPython.display import display, Markdown
print('\n--- Diagnóstico dos arquivos problemáticos ---')
if arquivos_com_erro:
    print('\nArquivos com erro de leitura:')
    for nome, erro in arquivos_com_erro:
        print(f'Arquivo: {nome} | Erro: {erro}')
        try:
            caminho = [arq for arq in arquivos_csv if arq.name == nome][0]
            with open(caminho, 'r', encoding='utf-8') as f:
                linhas = [next(f) for _ in range(5)]
            print('Primeiras linhas:')
            for l in linhas: print(l.strip())
        except Exception as e:
            print(f'Não foi possível ler as primeiras linhas: {e}')
if arquivos_colunas_incompativeis:
    print('\nArquivos com linhas ignoradas por colunas incompatíveis:')
    for nome in arquivos_colunas_incompativeis:
        print(f'Arquivo: {nome}')
        try:
            caminho = [arq for arq in arquivos_csv if arq.name == nome][0]
            with open(caminho, 'r', encoding='utf-8') as f:
                linhas = [next(f) for _ in range(5)]
            print('Primeiras linhas:')
            for l in linhas: print(l.strip())
        except Exception as e:
            print(f'Não foi possível ler as primeiras linhas: {e}')
if not arquivos_com_erro and not arquivos_colunas_incompativeis:
    print('Nenhum arquivo problemático detectado.')


--- Diagnóstico dos arquivos problemáticos ---

Arquivos com linhas ignoradas por colunas incompatíveis:
Arquivo: saida_queda_livre_dt_0_001.csv
Primeiras linhas:
tempo,posicao,velocidade,aceleracao
0.0000,100.0000,0.0000,-9.8100
0.0010,100.0000,-0.0098,-9.8100
0.0020,100.0000,-0.0196,-9.8100
0.0030,100.0000,-0.0294,-9.8100
Arquivo: saida_queda_livre_dt_0_01.csv
Primeiras linhas:
tempo,posicao,velocidade,aceleracao
0.0000,100.0000,0.0000,-9.8100
0.0100,100.0000,-0.0981,-9.8100
0.0200,99.9990,-0.1962,-9.8100
0.0300,99.9971,-0.2943,-9.8100
Arquivo: saida_queda_livre_dt_0_1.csv
Primeiras linhas:
tempo,posicao,velocidade,aceleracao
0.0000,100.0000,0.0000,-9.8100
0.1000,100.0000,-0.9810,-9.8100
0.2000,99.9019,-1.9620,-9.8100
0.3000,99.7057,-2.9430,-9.8100
Arquivo: saida_queda_livre_dt_0_5.csv
Primeiras linhas:
tempo,posicao,velocidade,aceleracao
0.0000,100.0000,0.0000,-9.8100
0.5000,100.0000,-4.9050,-9.8100
1.0000,97.5475,-9.8100,-9.8100
1.5000,92.6425,-14.7150,-9.8100
Arquivo: saida_queda

## 5. Gerar tabela em formato LaTeX
Vamos gerar a tabela em formato LaTeX usando o método `to_latex()` do pandas.

In [97]:
# Gera tabela LaTeX a partir do DataFrame consolidado (df)
from pathlib import Path
import pandas as pd

def escapar_latex(texto):
    # Escapa _ e ^ para LaTeX
    return str(texto).replace('_', '\\_').replace('^', '\\^{}')

tabelas_dir = Path('../relatorioCefet/tabelas').resolve()
tabelas_dir.mkdir(parents=True, exist_ok=True)

if 'df' in locals():
    # Escapa _ e ^ nos cabeçalhos
    df_latex = df.copy()
    df_latex.columns = [escapar_latex(col) for col in df_latex.columns]
    # Escapa _ e ^ em todo o conteúdo
    for col in df_latex.columns:
        df_latex[col] = df_latex[col].astype(str).apply(escapar_latex)
    caption = 'Tabela consolidada de todos os arquivos CSV da pasta dados'
    label = 'tab:consolidado_csv'
    tabela_latex = df_latex.to_latex(index=False, caption=caption, label=label, escape=False)
    caminho_saida = tabelas_dir / 'consolidado_csv.tex'
    with open(caminho_saida, 'w', encoding='utf-8') as f:
        f.write(tabela_latex)
    print(f'Tabela LaTeX salva em: {caminho_saida}')
    print(tabela_latex)
else:
    print('DataFrame df não encontrado. Execute as células anteriores para consolidar os dados.')

Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\relatorioCefet\tabelas\consolidado_csv.tex
\begin{table}
\caption{Tabela consolidada de todos os arquivos CSV da pasta dados}
\label{tab:consolidado_csv}
\begin{tabular}{llllll}
\toprule
dt & tempo\_final & v\_final\_num & v\_final\_analitico & erro\_relativo & arquivo\_origem \\
\midrule
0.001 & 4.483 & -44.978669 & 44.978899 & 1.999995 & acuracia\_com\_resistencia\_c\_0\_01.csv \\
0.011 & 4.488 & -45.027667 & 45.0302 & 1.999944 & acuracia\_com\_resistencia\_c\_0\_01.csv \\
0.021 & 4.494 & -45.086922 & 45.091763 & 1.999893 & acuracia\_com\_resistencia\_c\_0\_01.csv \\
0.031 & 4.526 & -45.412966 & 45.420165 & 1.999841 & acuracia\_com\_resistencia\_c\_0\_01.csv \\
0.041 & 4.51 & -45.246465 & 45.255951 & 1.99979 & acuracia\_com\_resistencia\_c\_0\_01.csv \\
0.051 & 4.539 & -45.541731 & 45.553608 & 1.999739 & acuracia\_com\_resistencia\_c\_0\_01.csv \\
0.061 & 4.514 & -45.282878 & 45.297002 & 1.999688 &

## 7. Copiar gráficos para a pasta de figuras do relatório

Esta etapa copia automaticamente todas as imagens geradas (gráficos) da pasta `src/graficos` para a pasta `relatorioCefet/figuras`, facilitando a inclusão dos gráficos no relatório final em LaTeX. Execute a célula abaixo para garantir que todos os arquivos de imagem estejam disponíveis na pasta correta para o seu documento.

## 8. Copiar toda a pasta de gráficos para figuras

Esta célula copia recursivamente todo o conteúdo da pasta `graficos` (incluindo subpastas e arquivos) para a pasta `relatorioCefet/figuras`. Útil para garantir que todos os gráficos estejam disponíveis no relatório.

In [98]:
import shutil
from pathlib import Path

graficos_dir = Path('../src/graficos').resolve()
figuras_dir = Path('../relatorioCefet/figuras').resolve()
destino_graficos = figuras_dir / 'graficos'

def copiar_pasta(origem, destino):
    if not origem.exists():
        print(f'Pasta de origem não existe: {origem}')
        return
    destino.mkdir(parents=True, exist_ok=True)
    for item in origem.iterdir():
        dest_item = destino / item.name
        if item.is_dir():
            copiar_pasta(item, dest_item)
        else:
            shutil.copy2(item, dest_item)
            print(f'Arquivo copiado: {item} -> {dest_item}')

copiar_pasta(graficos_dir, destino_graficos)

Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\src\graficos\acuracia_vfinal_com_resistencia_c_0_01.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\relatorioCefet\figuras\graficos\acuracia_vfinal_com_resistencia_c_0_01.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\src\graficos\acuracia_vfinal_com_resistencia_c_0_1.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\relatorioCefet\figuras\graficos\acuracia_vfinal_com_resistencia_c_0_1.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\src\graficos\acuracia_vfinal_com_resistencia_c_10.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\relatorioCefet\figuras\graficos\acuracia_vfinal_com_resistencia_c_10.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\pratica01\src\graficos\acuracia_vfinal_com_resistencia_c_2.png -> D:\GitHub\DoutoradoCefet\Princip